# PlantHelixSeek CRE Sliding-Window Scan (Arabidopsis showcase)

**Locus (illustrative):** `cre_locus=Chr1:5100001-5300000` — a 200 kb *Arabidopsis thaliana* Chr1 fragment (TAIR10).

**Methodology:** every scan, decode, peak-calling and metric rule in this notebook is transcribed
verbatim from the frozen Phase-6 locus-selection contract,
[`../plant_helixseek_shared/data/selection.md`](../plant_helixseek_shared/data/selection.md)
(the single source of truth for thresholds, tolerance bands and decode rules — nothing below is
re-derived or "improved").

**Frozen CRE contract summary (selection.md, "Verification contracts")**

- Model access through the dnallm public route only: `load_model_and_tokenizer(repo_id, TaskConfig, source="modelscope")`; the packaged registry (`dnallm/models/model_info.yaml`) is the single source of the repo id and label order. Forwards run under `torch.no_grad()`.
- Scan: 500 bp windows / 50 stride / 50 bp bins at batch 4 (the eager-attention ceiling through the dnallm route; batch 64 OOMs — never copy upstream's sdpa-assumed batch 256).
- Bin score: arithmetic mean of class-1 p(CRE) probabilities over the windows covering each bin; every bin is covered by at least one window (asserted below).
- Peak calling: threshold = mean + 1.5 sigma of the LOCUS bin scores; runs of above-threshold bins merge when separated by <= 50 bp (merge_gap); merged peaks shorter than 50 bp are dropped (min_length). Upstream's `min_score 0.6` / `max_length 5000` defaults are NOT applied.
- Metric: `bedtools jaccard` (v2.31.1) on coordinate-sorted BEDs in 0-based half-open coordinates (all conversions via `dnallm.utils.genomic_coords`), parsing the 3rd output column.

**Thresholds and tolerance bands (copied from selection.md):**

| Metric | Selection threshold | Tolerance band (assertion) | Observed (selection run) |
|---|---|---|---|
| CRE jaccard | >= 0.3 | [0.3, 1.00] | 0.3247 |
| Negative CRE peak-base fraction (flanking) | <= 0.05 | [0.00, 0.05] | 0.0325 |

**Environment:** the exact transformers / torch / flash-linear-attention versions this notebook ran
with are printed by the first code cell as `transformers_version=`, `torch_version=` and
`fla_version=` key=value lines.

**Disclaimer (illustrative loci):** all results in this notebook are computed on ONE illustrative
200 kb locus (plus one 20 kb flanking negative-control window) selected by the Phase-6 methodology.
They demonstrate the workflow on an illustrative locus, not genome-wide accuracy — see
selection.md for the selection methodology and the meaning of each tolerance band.

In [1]:
# Hard environment guard (D-16): the PlantHelixSeek remote code silently falls
# back to a pure-PyTorch non-KDA path when flash-linear-attention is absent,
# producing positionally-dead outputs -- absence is a hard error, never a
# silent degradation. The spec probe is deliberate (never a bare module
# import): the fast-lane test tests/examples/test_examples.py execs every
# import statement in this notebook on legs without the fla extra.
import importlib.metadata
import importlib.util

import torch
import transformers

if importlib.util.find_spec("fla") is None:
    raise RuntimeError(
        "flash-linear-attention (fla) is not installed: the PlantHelixSeek "
        "kernels would silently fall back to the non-KDA path and produce "
        "positionally-dead outputs (see README section Flash-Linear-Attention). "
        "Install with: uv pip install -e '.[fla]'"
    )

# The fla version is read via importlib.metadata.version -- a call, not an
# import statement, so no import node ever names the fla module.
print(f"transformers_version={transformers.__version__}")
print(f"torch_version={torch.__version__}")
print(f"fla_version={importlib.metadata.version('flash-linear-attention')}")
print(f"cuda_device={torch.cuda.get_device_name(0)}")

transformers_version=5.17.0
torch_version=2.11.0+cu130
fla_version=0.5.2
cuda_device=NVIDIA GB10


## Load the checkpoint through the dnallm public route

The repo id and label order come from the packaged registry entry for
`zhangtaolab/PlantHelixSeek-CRE` (frozen in Phase 6 with the checkpoint commit sha recorded in
`dnallm/models/model_info.yaml`); the checkpoint is fetched from ModelScope through the generic
`load_model_and_tokenizer` dispatch (no special handler).

In [2]:
from pathlib import Path

import dnallm
import yaml
from dnallm.configuration.configs import TaskConfig
from dnallm.models.model import load_model_and_tokenizer

REPO_ID = "zhangtaolab/PlantHelixSeek-CRE"

# Same registry-read shape as tests/models/test_plant_helixseek_smoke.py:
# the packaged yaml is the single source of repo id + label order.
registry_path = Path(dnallm.__file__).parent / "models" / "model_info.yaml"
registry = yaml.safe_load(registry_path.read_text(encoding="utf-8"))
entries = [e for e in registry["finetuned"] if e.get("model") == REPO_ID]
assert len(entries) == 1, f"expected exactly one registry entry for {REPO_ID}"
task = entries[0]["task"]
print(f"registry_task_type={task['task_type']}")
print(f"registry_num_labels={task['num_labels']}")
print(f"registry_label_names={task['label_names']}")

task_config = TaskConfig(
    task_type=task["task_type"],
    num_labels=task["num_labels"],
    label_names=task["label_names"],
    threshold=task["threshold"],
)
model, tokenizer = load_model_and_tokenizer(REPO_ID, task_config, source="modelscope")
print(f"model_id2label={model.config.id2label}")
print(f"model_device={model.device}")

registry_task_type=binary

registry_num_labels=2

registry_label_names=['Not CRE', 'CRE']

22:10:55 - dnallm.utils.support - 

INFO

 - Model files are stored in /home/forrest/.cache/modelscope/hub/models/zhangtaolab/PlantHelixSeek-CRE


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

model_id2label={0: 'Not CRE', 1: 'CRE'}

model_device=cuda:0

## Load the committed showcase data

The 200 kb locus FASTA carries its genomic interval in the header — the genomic offset is parsed
from that header (never hardcoded a second time). The PlantDHS truth slice is read DIRECTLY: the
committed file already holds the fully-contained truth rows verbatim (never re-sliced — the
`slice_gff_rows` helper uses overlap semantics), converted to 0-based half-open genomic
coordinates via `gff1_to_half_open`.

In [3]:
import re

from dnallm.utils import fetch_sequence, gff1_to_half_open

# Fragment-local coordinates: the committed .fas holds exactly the locus.
fasta_path = Path("data/chr1_5100001_5300000.fas")
header = fasta_path.read_text(encoding="utf-8").splitlines()[0]
locus_match = re.search(r"\[(\d+),\s*(\d+)\]", header)
locus_start, locus_end = int(locus_match.group(1)), int(locus_match.group(2))
# Genomic offset (single source: the FASTA header interval, 1-based closed).
genomic_offset = locus_start - 1
locus_length = locus_end - locus_start + 1
print(f"cre_locus=Chr1:{locus_start}-{locus_end}")

sequence = fetch_sequence(str(fasta_path), "Chr1", 1, locus_length)
assert len(sequence) == locus_length

# Truth: the committed PlantDHS slice, verbatim rows, source order preserved.
truth_gff = Path("data/TAIR10_DHSs_chr1_5100001_5300000.gff")
truth_rows = [row for row in truth_gff.read_text(encoding="utf-8").splitlines() if row.strip()]
assert truth_rows, "truth DHS slice is empty"
# Truth rows are ALREADY genomic (1-based closed, e.g. 5102240): only the
# FASTA fragment is local, so the half-open conversion takes no offset.
truth_intervals = []
for row in truth_rows:
    cols = row.split("\t")
    start0, end0 = gff1_to_half_open(int(cols[3]), int(cols[4]))
    truth_intervals.append((start0, end0))
print(f"truth_dhs_rows={len(truth_rows)}")

cre_locus=Chr1:5100001-5300000

truth_dhs_rows=94

## Sliding-window scan (500 bp / 50 stride / 50 bp bins, batch 4)

Every window is scored p(CRE) = softmax(logits)[:, 1] under `torch.no_grad()` (mandatory — autograd
over these windows OOMs the box). The per-bin score is the arithmetic mean of the class-1
probabilities of the windows covering the bin; the frozen contract guarantees and asserts that every
bin is covered by at least one window.

In [4]:
import numpy as np

# Frozen scan contract (selection.md): 500/50/50 at batch 4 -- the
# eager-attention ceiling through the dnallm route (dnallm forces
# attn_implementation="eager"; upstream's sdpa-assumed batch 256 OOMs:
# bs=64 measured 30.8 GiB in a single allocation).
window, stride, bin_width, batch_size = 500, 50, 50, 4


def scan_bin_scores(seq):
    """Run the frozen 500/50/50 sliding scan at batch 4; return per-bin mean p(CRE)."""
    starts = list(range(0, len(seq) - window + 1, stride))
    probabilities = []
    with torch.no_grad():
        for i in range(0, len(starts), batch_size):
            batch = [seq[s : s + window] for s in starts[i : i + batch_size]]
            enc = tokenizer(batch, return_tensors="pt", padding=True)
            out = model(
                input_ids=enc["input_ids"].to(model.device),
                attention_mask=enc["attention_mask"].to(model.device),
            )
            p_cre = torch.softmax(out.logits, dim=-1)[:, 1]
            probabilities.extend(p_cre.float().cpu().tolist())
    probabilities = np.asarray(probabilities)

    n_bins = len(seq) // bin_width
    scores = np.zeros(n_bins)
    counts = np.zeros(n_bins, dtype=int)
    for start, p in zip(starts, probabilities):
        scores[start // bin_width : (start + window) // bin_width] += p
        counts[start // bin_width : (start + window) // bin_width] += 1
    # Frozen structural assert: every bin is covered by at least one window.
    assert counts.min() >= 1, f"bin coverage gap: min coverage {counts.min()}"
    # Bin score = arithmetic mean of class-1 probabilities over covering windows.
    return scores / counts


bin_scores = scan_bin_scores(sequence)
print(f"scan_windows={(len(sequence) - window) // stride + 1}")
print(f"scan_bins={len(bin_scores)}")
print(f"bin_score_mean={bin_scores.mean():.4f}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


HelixSeek requires an initialized cache to return a cache. None was provided, so no cache will be returned.


scan_windows=3991

scan_bins=4000

bin_score_mean=0.3180

## Peak calling (mean + 1.5 sigma, merge_gap 50, min_length 50)

The threshold is calibrated on THIS locus's bin scores; above-threshold bin runs merge when
separated by at most 50 bp and merged peaks shorter than 50 bp are dropped. Peaks are written as a
coordinate-sorted 0-based half-open narrowPeak (BED6 form, signalValue = mean bin score) in
GENOMIC coordinates under `outputs/` (gitignored).

In [5]:
# Frozen peak-calling rule (selection.md); upstream's min_score 0.6 /
# max_length 5000 defaults are NOT applied.
merge_gap, min_length = 50, 50
threshold = bin_scores.mean() + 1.5 * bin_scores.std()
print(f"peak_threshold={threshold:.4f}")


def call_peaks(scores, peak_threshold):
    """Above-threshold bin runs -> merge_gap merge -> min_length filter (frozen rule)."""
    above = scores > peak_threshold
    runs = []
    run_start = None
    for i, is_above in enumerate(above):
        if is_above and run_start is None:
            run_start = i
        elif not is_above and run_start is not None:
            runs.append((run_start, i))
            run_start = None
    if run_start is not None:
        runs.append((run_start, len(above)))

    merged = []
    for start_bin, end_bin in runs:
        if merged and (start_bin - merged[-1][1]) * bin_width <= merge_gap:
            merged[-1][1] = end_bin
        else:
            merged.append([start_bin, end_bin])

    return [
        (start_bin, end_bin)
        for start_bin, end_bin in merged
        if (end_bin - start_bin) * bin_width >= min_length
    ]


def peaks_to_narrowpeak(peak_bins, scores, offset, chrom="Chr1"):
    """Write peaks as sorted 0-based half-open GENOMIC narrowPeak rows (BED6 core)."""
    rows = []
    for n_peak, (start_bin, end_bin) in enumerate(peak_bins, start=1):
        genomic_start = start_bin * bin_width + offset
        genomic_end = end_bin * bin_width + offset
        signal = float(scores[start_bin:end_bin].mean())
        rows.append(
            f"{chrom}\t{genomic_start}\t{genomic_end}\tCRE_peak_{n_peak}\t"
            f"{min(1000, int(round(signal * 1000)))}\t.\t{signal:.4f}\t-1\t-1\t-1"
        )
    return rows


peak_bins = call_peaks(bin_scores, threshold)
predicted_rows = peaks_to_narrowpeak(peak_bins, bin_scores, genomic_offset)
predicted_peak_bases = sum((end - start) * bin_width for start, end in peak_bins)
truth_rows_bed = [
    f"Chr1\t{start0}\t{end0}\tDHS_{i}\t0\t."
    for i, (start0, end0) in enumerate(sorted(truth_intervals), start=1)
]

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)
predicted_bed_path = outputs_dir / "predicted_cre_peaks.narrowPeak"
predicted_bed_path.write_text("\n".join(predicted_rows) + "\n", encoding="utf-8")
truth_bed_path = outputs_dir / "truth_dhs.bed"
truth_bed_path.write_text("\n".join(truth_rows_bed) + "\n", encoding="utf-8")
print(f"predicted_peaks={len(predicted_rows)}")
print(f"predicted_peak_bases={predicted_peak_bases}")
print(f"truth_dhs_bases={sum(end - start for start, end in truth_intervals)}")

peak_threshold=0.7124

predicted_peaks=51

predicted_peak_bases=19750

truth_dhs_bases=45000

## Jaccard against the PlantDHS truth

`bedtools jaccard` (v2.31.1) on the two coordinate-sorted 0-based half-open BEDs; the jaccard
index is the 3rd output column.

In [6]:
import subprocess

result = subprocess.run(
    ["bedtools", "jaccard", "-a", str(predicted_bed_path), "-b", str(truth_bed_path)],
    capture_output=True,
    text=True,
    check=True,
)
jaccard_value = float(result.stdout.strip().splitlines()[1].split("\t")[2])
print(f"jaccard={jaccard_value:.4f}")

jaccard=0.3247

## Prediction vs truth track view

Predicted per-bin CRE score (top) and PlantDHS open-chromatin coverage (bottom) across the locus,
downsampled to ~1 point per bin (~4000 points per track so the embedded vega output stays small).

In [7]:
import json

import altair as alt
import pandas as pd
import vl_convert as vlc
from IPython.display import display

alt.data_transformers.enable("default", max_rows=None)

bin_starts_genomic = (np.arange(len(bin_scores)) * bin_width + genomic_offset).tolist()

# Truth coverage per bin: fraction of each 50 bp bin covered by a PlantDHS row.
truth_covered_bases = np.zeros(len(bin_scores), dtype=int)
for start0, end0 in truth_intervals:
    b0 = max((start0 - genomic_offset) // bin_width, 0)
    b1 = min(-(-(end0 - genomic_offset) // bin_width), len(bin_scores))
    for b in range(b0, b1):
        covered = min(end0 - genomic_offset, (b + 1) * bin_width) - max(
            start0 - genomic_offset, b * bin_width
        )
        truth_covered_bases[b] += covered

pred_df = pd.DataFrame(
    {"start": bin_starts_genomic, "score": np.round(bin_scores, 4).tolist()}
)
truth_df = pd.DataFrame(
    {
        "start": bin_starts_genomic,
        "end": [s + bin_width for s in bin_starts_genomic],
        "dhs": (truth_covered_bases / bin_width).round(3).tolist(),
    }
)

prediction_track = (
    alt.Chart(pred_df)
    .mark_line(color="#4c78a8")
    .encode(
        x=alt.X("start:Q").title("Chr1 position (bp)").axis(format="s"),
        y=alt.Y("score:Q").title("predicted p(CRE) bin score").scale(zero=False),
    )
)
truth_track = (
    alt.Chart(truth_df)
    .mark_rect(color="#f58518")
    .encode(
        x=alt.X("start:Q").title(None),
        x2=alt.X2("end:Q"),
        y=alt.Y("dhs:Q").title("PlantDHS truth coverage").scale(domain=[0, 1]),
        y2=alt.value(0),
    )
)
track_chart = alt.vconcat(prediction_track, truth_track).resolve_scale(x="shared")

# Embed the figure as inline vega JSON (no external files, no binaries): the
# compiled vega spec renders in the GitHub notebook viewer and the native
# vega-lite spec rides along for newer front-ends. Downsampling to ~1 point
# per bin keeps the embedded payloads well inside the 2 MB notebook budget.
vegalite_spec = json.loads(track_chart.to_json())
vega_spec = vlc.vegalite_to_vega(vegalite_spec)
display(
    {
        # Compiled vega (object form: the canonical embedded-output shape).
        "application/vnd.vega.v6+json": vega_spec,
        # Native vega-lite as a JSON string (nbformat allows objects only
        # under +json mimes; a bare .json mime must carry a string).
        "application/vnd.vegalite.v6.json": json.dumps(vegalite_spec),
        "text/plain": (
            "PlantHelixSeek CRE predicted bin-score track (top) vs "
            "PlantDHS truth track (bottom)"
        ),
    },
    raw=True,
)

PlantHelixSeek CRE predicted bin-score track (top) vs PlantDHS truth track (bottom)

Predicted CRE signal (top, blue) broadly tracks the PlantDHS open-chromatin regions
(bottom, orange) on this locus; called peaks are the mean + 1.5 sigma excursions of the top track.
Illustrative locus, not genome-wide accuracy.

## Negative control: flanking low-signal window

Phase 6 selected a 20 kb flanking window (`Chr1:5351001-5371000`, the lowest PlantDHS row count
within ±100 kb of the CRE locus, not overlapping it; the committed truth slice
[`../plant_helixseek_shared/data/TAIR10_DHSs_chr1_5351001_5371000.gff`](../plant_helixseek_shared/data/TAIR10_DHSs_chr1_5351001_5371000.gff)
holds 3 DHS rows) as the expected near-zero-signal control. The assertion metric is the
predicted-peak base fraction at the CRE-locus-calibrated ABSOLUTE threshold — never a
jaccard-against-empty (a degenerate 0/0). The selection run recorded
`neg_cre_fraction=0.0325` (1 peak); the selection.md tolerance band is [0.00, 0.05].

In [8]:
flank_fasta = Path("../plant_helixseek_shared/data/chr1_5351001_5371000.fas")
flank_header = flank_fasta.read_text(encoding="utf-8").splitlines()[0]
flank_match = re.search(r"\[(\d+),\s*(\d+)\]", flank_header)
flank_start, flank_end = int(flank_match.group(1)), int(flank_match.group(2))
flank_offset = flank_start - 1
flank_length = flank_end - flank_start + 1
flank_sequence = fetch_sequence(str(flank_fasta), "Chr1", 1, flank_length)
print(f"flanking_locus=Chr1:{flank_start}-{flank_end}")

# Same frozen 500/50/50 scan; scored against the ABSOLUTE threshold
# calibrated on the CRE locus above (never re-calibrated on this window).
flank_bin_scores = scan_bin_scores(flank_sequence)
flank_peak_bins = call_peaks(flank_bin_scores, threshold)
flank_peak_bases = sum((end - start) * bin_width for start, end in flank_peak_bins)
neg_cre_fraction_value = flank_peak_bases / flank_length
print(f"neg_cre_fraction={neg_cre_fraction_value:.4f}")
print(f"flanking_peak_count={len(flank_peak_bins)}")

flanking_locus=Chr1:5351001-5371000

neg_cre_fraction=0.0325

flanking_peak_count=1

## Observed vs the frozen contract

`selection.md` is parsed at runtime (line-start key anchors). This notebook PRINTS the comparison
and asserts nothing on metric floors — the nightly test layer owns every assertion.

In [9]:
selection_text = Path("../plant_helixseek_shared/data/selection.md").read_text(
    encoding="utf-8"
)
frozen = {}
for key in ("jaccard", "genes_above_floor", "neg_cre_fraction", "neg_anno_fraction"):
    frozen_match = re.search(rf"^{key}=([0-9.]+)", selection_text, re.MULTILINE)
    if frozen_match is None:
        raise RuntimeError(f"selection.md is missing frozen key '{key}=' (parse guard)")
    frozen[key] = float(frozen_match.group(1))

bands = {}
for metric_label in (
    "CRE jaccard",
    "Negative CRE peak-base fraction (flanking)",
    "Negative Anno genic-base fraction (intergenic)",
):
    for line in selection_text.splitlines():
        if line.startswith(f"| {metric_label} |"):
            band_match = re.search(r"\[([0-9.]+),\s*([0-9.]+)\]", line)
            if band_match is None:
                raise RuntimeError(
                    f"selection.md band row '{metric_label}' has no '[a, b]' interval (parse guard)"
                )
            bands[metric_label] = (float(band_match.group(1)), float(band_match.group(2)))
            break

comparison = [
    ("CRE jaccard", jaccard_value, frozen["jaccard"]),
    (
        "Negative CRE peak-base fraction (flanking)",
        neg_cre_fraction_value,
        frozen["neg_cre_fraction"],
    ),
]
print(f"{'metric':<46}{'observed':>10}{'selection':>11}{'band':>16}")
for metric_label, observed_value, selection_value in comparison:
    low, high = bands[metric_label]
    inside = low <= observed_value <= high
    print(
        f"{metric_label:<46}{observed_value:>10.4f}{selection_value:>11.4f}"
        f"{'[' + format(low, '.2f') + ', ' + format(high, '.2f') + ']':>16}"
        f"{'inside' if inside else 'OUTSIDE'}"
    )

metric                                          observed  selection            band

CRE jaccard                                       0.3247     0.3247    [0.30, 1.00]inside

Negative CRE peak-base fraction (flanking)        0.0325     0.0325    [0.00, 0.05]inside

## Summary

On the illustrative locus Chr1:5100001-5300000 the frozen-contract scan + peak calling +
`bedtools jaccard` pipeline reproduces the Phase-6 selection behavior (`jaccard=` and
`neg_cre_fraction=` stream lines above), and every rule used here is the frozen Phase-6 selection
contract — see [`../plant_helixseek_shared/data/selection.md`](../plant_helixseek_shared/data/selection.md).
These results are computed on one illustrative locus, not genome-wide accuracy; the nightly
execution tests assert the tolerance bands parsed from that same document.